# Phase 2N v2 - Deduplication against a distribution of random deletions

**Run on:** Kaggle, free GPU. About 8.5 hours; resumable across sessions.

---

### What changed from v1, and why

v1 paired each deduplicated training set with **one** size-matched random
deletion per fold. A single draw is noisy: which images happen to be removed
moves balanced accuracy by about 0.05 on its own. v2 repeats the random deletion
**20 times per fold** and retrains the (fixed) deduplicated set under **5 seeds**,
so each fold's dedup score is compared against a distribution of the loss a
smaller training set causes on its own.

| arm | training set | test set | replicates per fold |
|---|---|---|---|
| baseline | full | fold | 1 |
| **dedup** | test-neighbours removed | identical | 5 seeds |
| random | same number removed at random | identical | 20 draws |

### Analysis rule, fixed before the run

Notation, per condition and fold $k$: $\bar D_k$ the mean over dedup seeds,
$\bar R_k$ the mean over random draws.

1. **Leakage estimate** per fold: $e_k = \bar R_k - \bar D_k$, the observed
   dedup loss minus the loss expected from a smaller training set. Reported:
   the mean over folds with a Nadeau-Bengio corrected 95% interval
   ($\rho = n_\text{test}/n_\text{train}$ of the deduplicated set, $t$ with 4
   df), exactly as v1, plus each fold's percentile of $\bar D_k$ within its own
   random-deletion distribution.
2. **Holm** across the three conditions. A condition is "dedup costs more than
   size-matched random deletion" if its Holm-adjusted $p < 0.05$ and the interval
   lies above zero. Otherwise it is reported as not established.
3. **v2 replaces v1** in the paper for these conditions, whichever direction it
   moves. No condition is dropped after the run.
4. The analysis cell runs only when every planned run is present. No interim
   effect is printed while runs are incomplete.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "kagglehub", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, json, time, shutil, random, gc, glob
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
RESULTS_DIR = f"{WORK}/fyp_phase2nr_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
ROWS_CSV = f"{RESULTS_DIR}/phase2nr_rows.csv"

N_FOLDS = 5
M_DEDUP, N_RANDOM = 5, 20
CONDITIONS = [("ResNet50 (pretrained)", 0.98),
              ("EfficientNetB0 (pretrained)", 0.98),
              ("EfficientNetB0 (pretrained)", 0.94)]
TIME_BUDGET_MIN = 11 * 60          # stop launching new runs after this
T_START = time.time()

RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "m_dedup": M_DEDUP,
           "n_random": N_RANDOM, "conditions": CONDITIONS,
           "tf": tf.__version__}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)

In [ ]:
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    DATA_ROOT = os.path.dirname([d for d, _, _ in os.walk(DL)
                                 if os.path.basename(d) == "Malignant cases"][0])

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{WORK}/split_seed42.csv", SPLIT_URL], check=True)
df = pd.read_csv(f"{WORK}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])
y_all = df["y"].to_numpy()

thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean(); n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs); S = T @ T.T; np.fill_diagonal(S, 0.0)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df, y_all))
print("images", len(df))

## 2. Training sets for every arm

Every draw is seeded by (condition, fold, arm, replicate), so the sets are the
same however the runs are scheduled or resumed.

In [ ]:
ARM_ID = {"baseline": 0, "dedup": 1, "random": 2}

def training_set(ci, k, arm, j):
    arch, thr = CONDITIONS[ci]
    tr, te = folds[k]
    contaminated = (S[np.ix_(tr, te)] >= thr).any(axis=1)
    rng = np.random.default_rng([SEED, ci, k, ARM_ID[arm], j])
    info = {"n_removed": int(contaminated.sum())}
    if arm == "baseline":
        return tr, info
    if arm == "dedup":
        return tr[~contaminated], info
    if arm == "random":
        keep = rng.choice(len(tr), size=len(tr) - int(contaminated.sum()), replace=False)
        return np.sort(tr[keep]), info
    raise ValueError(arm)

PLAN = []
for j in range(max(M_DEDUP, N_RANDOM)):
    for ci, (arch, thr) in enumerate(CONDITIONS):
        for k in range(N_FOLDS):
            if j == 0 and not any(c[0] == arch for c in CONDITIONS[:ci]):
                PLAN.append((ci, k, "baseline", 0))
            if j < M_DEDUP:
                PLAN.append((ci, k, "dedup", j))
            if j < N_RANDOM:
                PLAN.append((ci, k, "random", j))
print("planned runs:", len(PLAN))

# sanity: sizes match and test folds are untouched
for ci in range(len(CONDITIONS)):
    for k in range(N_FOLDS):
        d, _ = training_set(ci, k, "dedup", 0)
        r, _ = training_set(ci, k, "random", 0)
        assert len(d) == len(r)
        assert not set(d) & set(folds[k][1])
        assert len(d) == len(folds[k][0]) - _["n_removed"]
print("training-set sizes matched, test folds untouched: OK")
RESULTS["planned_runs"] = len(PLAN)
save_json()

## 3. Run (resumable)

In [ ]:
# Training runs in child processes of CHUNK runs each. The first GPU session hit
# ResourceExhaustedError after 155 runs in one process (8 Oct 2026); a fresh
# process per chunk returns all memory. The child trains exactly the training
# sets precomputed here; nothing about the design changes.
import pickle
SIZE, EPOCHS, BATCH, LR = 224, 30, 16, 1e-4
CHUNK = 30
TMP = "/kaggle/temp" if IN_KAGGLE else "/tmp"     # scratch, kept out of the outputs
os.makedirs(TMP, exist_ok=True)
X_PATH, JOBS_PATH = f"{TMP}/X_{SIZE}_uint8.npy", f"{TMP}/jobs.pkl"
COLS = ["ci", "architecture", "threshold", "fold", "arm", "rep", "seed", "n_train",
        "n_test", "n_removed", "accuracy", "balanced_accuracy", "macro_f1", "minutes",
        "max_rss_gb"]
X = np.empty((len(df), SIZE, SIZE, 3), np.uint8)   # same pixel values, 1/4 the memory
for i, p in enumerate(tqdm(df["path"], desc=f"load {SIZE}px")):
    X[i] = np.asarray(Image.open(p).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR))
np.save(X_PATH, X); del X; gc.collect()

def key(r):
    return (int(r["ci"]), int(r["fold"]), r["arm"], int(r["rep"]))

done = {}
for f in glob.glob("/kaggle/input/**/phase2nr_rows.csv", recursive=True) +          ([ROWS_CSV] if os.path.exists(ROWS_CSV) else []):
    for r in pd.read_csv(f).to_dict("records"):
        done[key(r)] = r
print("runs carried over from earlier sessions:", len(done))
pd.DataFrame(list(done.values()), columns=COLS).to_csv(ROWS_CSV, index=False)

jobs = []
for (ci, k, arm, j) in PLAN:
    arch, thr = CONDITIONS[ci]
    tr, info = training_set(ci, k, arm, j)
    jobs.append({"ci": ci, "architecture": arch, "threshold": thr, "fold": k, "arm": arm,
                 "rep": j, "tr": tr, "te": folds[k][1], "n_removed": info["n_removed"],
                 "seed": SEED + 10000 * ci + 1000 * k + 100 * ARM_ID[arm] + j})
with open(JOBS_PATH, "wb") as fh:
    pickle.dump({"jobs": jobs, "y": y_all, "cols": COLS}, fh)

CHILD = r'''
import sys, os, gc, time, pickle, resource
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)
X_PATH, JOBS_PATH, ROWS_CSV, SIZE, EPOCHS, BATCH, LR, CHUNK = sys.argv[1:9]
SIZE, EPOCHS, BATCH, CHUNK, LR = int(SIZE), int(EPOCHS), int(BATCH), int(CHUNK), float(LR)
X = np.load(X_PATH, mmap_mode="r")
d = pickle.load(open(JOBS_PATH, "rb")); jobs, y_all, COLS = d["jobs"], d["y"], d["cols"]
done = set()
if os.path.exists(ROWS_CSV):
    for r in pd.read_csv(ROWS_CSV).to_dict("records"):
        done.add((int(r["ci"]), int(r["fold"]), r["arm"], int(r["rep"])))

def _transfer(base_fn, preprocess, unfreeze=20):
    inp = layers.Input((SIZE, SIZE, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = base_fn(include_top=False, weights="imagenet", input_shape=(SIZE, SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    x = base(preprocess(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(3, activation="softmax")(x))

BUILD = {
    "EfficientNetB0 (pretrained)": lambda: _transfer(
        keras.applications.EfficientNetB0,
        keras.applications.efficientnet.preprocess_input),
    "ResNet50 (pretrained)": lambda: _transfer(
        keras.applications.ResNet50, keras.applications.resnet50.preprocess_input),
}

n = 0
for jb in jobs:
    k4 = (jb["ci"], jb["fold"], jb["arm"], jb["rep"])
    if k4 in done:
        continue
    if n >= CHUNK:
        break
    tr, te = jb["tr"], jb["te"]
    keras.backend.clear_session(); tf.random.set_seed(jb["seed"])
    counts = np.bincount(y_all[tr], minlength=3)
    cw = {i: float(len(tr) / (3 * c)) if c else 0.0 for i, c in enumerate(counts)}
    model = BUILD[jb["architecture"]]()
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    model.fit(X[tr].astype(np.float32), y_all[tr], epochs=EPOCHS, batch_size=BATCH,
              class_weight=cw, verbose=0)
    pred = model.predict(X[te].astype(np.float32), verbose=0).argmax(1)
    row = {"ci": jb["ci"], "architecture": jb["architecture"], "threshold": jb["threshold"],
           "fold": jb["fold"], "arm": jb["arm"], "rep": jb["rep"], "seed": jb["seed"],
           "n_train": int(len(tr)), "n_test": int(len(te)), "n_removed": jb["n_removed"],
           "accuracy": float(accuracy_score(y_all[te], pred)),
           "balanced_accuracy": float(balanced_accuracy_score(y_all[te], pred)),
           "macro_f1": float(precision_recall_fscore_support(
               y_all[te], pred, labels=[0, 1, 2], average="macro", zero_division=0)[2]),
           "minutes": (time.time() - t0) / 60,
           "max_rss_gb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6}
    pd.DataFrame([row], columns=COLS).to_csv(ROWS_CSV, mode="a", index=False,
                                             header=not os.path.exists(ROWS_CSV))
    print(k4, f"bal {row['balanced_accuracy']:.3f}", f"rss {row['max_rss_gb']:.1f} GB",
          flush=True)
    del model; gc.collect(); n += 1
'''
with open(f"{TMP}/train_chunk.py", "w") as fh:
    fh.write(CHILD)

def n_missing():
    have = set()
    if os.path.exists(ROWS_CSV):
        have = {key(r) for r in pd.read_csv(ROWS_CSV).to_dict("records")}
    return sum(p not in have for p in PLAN)

stalls = 0
while n_missing() and (time.time() - T_START) / 60 < TIME_BUDGET_MIN:
    before = n_missing()
    r = subprocess.run([sys.executable, f"{TMP}/train_chunk.py", X_PATH, JOBS_PATH,
                        ROWS_CSV, str(SIZE), str(EPOCHS), str(BATCH), str(LR), str(CHUNK)])
    after = n_missing()
    print(f"chunk exit {r.returncode}: {before - after} runs done, {after} left",
          flush=True)
    stalls = stalls + 1 if after == before else 0
    if stalls >= 2:
        print("two chunks in a row made no progress; stopping."); break

done = {key(r): r for r in pd.read_csv(ROWS_CSV).to_dict("records")}

missing = [p for p in PLAN if p not in done]
RESULTS["complete"] = not missing
RESULTS["runs_done"] = len(PLAN) - len(missing)
print(f"{RESULTS['runs_done']} of {len(PLAN)} runs present")
save_json()

## 4. The answer (runs only when every planned run is present)

In [ ]:
from scipy import stats

def nb_interval(d, rho):
    k = len(d)
    se = np.sqrt((1.0 / k + rho) * d.var(ddof=1))
    crit = stats.t.ppf(0.975, k - 1)
    p = float(2 * stats.t.sf(abs(d.mean() / se), k - 1)) if se > 0 else float("nan")
    return float(d.mean()), float(d.mean() - crit * se), float(d.mean() + crit * se), p

def holm(ps):
    order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1.0, run)
    return adj.tolist()

if not RESULTS["complete"]:
    print("Incomplete. No effects are computed until every planned run is present.")
else:
    cv = pd.DataFrame(list(done.values()))
    cv.to_csv(ROWS_CSV, index=False)
    out, ps = [], []
    for ci, (arch, thr) in enumerate(CONDITIONS):
        sel = cv[cv.ci == ci]
        base = cv[(cv.architecture == arch) & (cv.arm == "baseline")]
        D = sel[sel.arm == "dedup"].groupby("fold")["balanced_accuracy"]
        R = sel[sel.arm == "random"].groupby("fold")["balanced_accuracy"]
        Dm, Rm = D.mean().to_numpy(), R.mean().to_numpy()
        rho = float((sel[sel.arm == "dedup"].n_test / sel[sel.arm == "dedup"].n_train).mean())
        e = Rm - Dm
        m, lo, hi, p = nb_interval(e, rho)
        pct = [float((sel[(sel.arm == "random") & (sel.fold == k)]
                      .balanced_accuracy <= Dm[k]).mean()) for k in range(N_FOLDS)]
        res = {"architecture": arch, "threshold": thr, "rho": rho,
               "baseline": float(base.balanced_accuracy.mean()),
               "dedup": float(Dm.mean()), "random": float(Rm.mean()),
               "random_sd_within_fold": float(sel[sel.arm == "random"]
                                              .groupby("fold").balanced_accuracy.std().mean()),
               "dedup_sd_within_fold": float(D.std().mean()),
               "effect_per_fold": e.tolist(), "effect": m, "ci": [lo, hi], "p": p,
               "dedup_percentile_in_random_null": pct}
        out.append(res); ps.append(p)
    for res, pa in zip(out, holm(ps)):
        res["p_holm"] = pa
        res["established"] = bool(pa < 0.05 and res["ci"][0] > 0)
        print(f"{res['architecture']} @ {res['threshold']}: dedup {res['dedup']:.4f} | "
              f"random {res['random']:.4f} (within-fold sd {res['random_sd_within_fold']:.3f})")
        print(f"   leakage estimate {res['effect']:+.4f}  95% CI ({res['ci'][0]:+.3f}, "
              f"{res['ci'][1]:+.3f})  p_holm {pa:.4f}  established: {res['established']}")
        print(f"   dedup percentile in its random null, per fold: "
              f"{[round(x, 2) for x in res['dedup_percentile_in_random_null']]}")
    RESULTS["verdict"] = out
    save_json()

    fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(5 * len(CONDITIONS), 4),
                             squeeze=False)
    for ax, (ci, (arch, thr)) in zip(axes[0], enumerate(CONDITIONS)):
        sel = cv[cv.ci == ci]
        for k in range(N_FOLDS):
            r = sel[(sel.arm == "random") & (sel.fold == k)].balanced_accuracy
            ax.violinplot([r], positions=[k], widths=0.7, showextrema=False)
            ax.scatter([k] * M_DEDUP, sel[(sel.arm == "dedup") & (sel.fold == k)]
                       .balanced_accuracy, color="#c0392b", s=18, zorder=3)
        ax.set_xticks(range(N_FOLDS)); ax.set_xlabel("fold")
        ax.set_title(f"{arch.split(' ')[0]} @ {thr}", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    axes[0][0].set_ylabel("balanced accuracy")
    fig.suptitle("Random-deletion null per fold (violins) against dedup seeds (red)",
                 fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(f"{RESULTS_DIR}/fig_dedup_null.png", dpi=200)

shutil.make_archive(f"{WORK}/fyp_phase2nr_results", "zip", RESULTS_DIR)
print("archived")